# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display summary description
print(f"{getattr(metadata, 'name', '<No Name>')}: {getattr(metadata, 'description', '<No Description>')}")

## 2. Data Overview
Review all available record sets and their fields. All entities are referenced by their `@id`.

In [ ]:
# Helper functions to get record sets and display their fields
record_sets = list(dataset.record_sets)

print("Available record sets and their fields (referenced by @id):\n")
record_set_ids = []
for rs in record_sets:
    r_id = rs.id
    record_set_ids.append(r_id)
    print(f'Record set @id: {r_id}')
    for field in rs.fields:
        print(f"  Field @id: {field.id} | name: {getattr(field, 'name', '<no name>')} | dataType: {getattr(field, 'data_type', '<no dataType>')} | column: {getattr(field.column, 'id', '<no column>') if hasattr(field, 'column') else '<no column>'}")
    print()

## 3. Data Extraction
Load data from one or more record sets into Pandas DataFrame(s) for analysis. Use the `@id` strings of the selected record sets.

In [ ]:
# Choose the record set(s) you want to extract data from
# Here we load all, but you can narrow to a subset if desired
dataframes = {}

for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"Loaded DataFrame for Record Set '@id': {record_set}")
        print("Columns:", df.columns.tolist())
        display(df.head())
    else:
        print(f"No records found for Record Set '@id': {record_set}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, categorizing data, etc.

Let's choose main protein abundance data (first non-empty record set automatically) for demo.

In [ ]:
# Pick a DataFrame with actual data
extracted_id = None
for rid, df in dataframes.items():
    if len(df) > 0:
        extracted_id = rid
        break

if extracted_id is None:
    raise RuntimeError("No non-empty record set available for analysis.")
print(f"Proceeding with Record Set '@id': {extracted_id}")

# List columns
print("Columns in this DataFrame:", dataframes[extracted_id].columns.tolist())

# Try to infer a numeric field from the columns (e.g. any whose dtype is float or int)
candidate_numeric = None
for col in dataframes[extracted_id].columns:
    if pd.api.types.is_numeric_dtype(dataframes[extracted_id][col]):
        candidate_numeric = col
        break
# If none found, try to convert one by regex (e.g. those with 'abundance', 'count', etc)
if candidate_numeric is None:
    for col in dataframes[extracted_id].columns:
        if any(x in col.lower() for x in ['abundance','count','coverage','mw','peptide','value']):
            # Try float conversion
            with pd.option_context('mode.use_inf_as_na', True):
                s = pd.to_numeric(dataframes[extracted_id][col], errors='coerce')
                if s.notna().sum() > 0:
                    dataframes[extracted_id][col] = s
                    candidate_numeric = col
                    break

if candidate_numeric is None:
    raise RuntimeError("No suitable numeric field found.")

print(f"Using '{candidate_numeric}' as example numeric field.")

# Try filtering
threshold = dataframes[extracted_id][candidate_numeric].mean() if dataframes[extracted_id][candidate_numeric].mean() == dataframes[extracted_id][candidate_numeric].mean() else 10
filtered_df = dataframes[extracted_id][dataframes[extracted_id][candidate_numeric] > threshold]
print(f"Filtered records with {candidate_numeric} > {threshold}:")
display(filtered_df.head())

# Normalize
norm_col = f"{candidate_numeric}_normalized"
filtered_df[norm_col] = (filtered_df[candidate_numeric] - filtered_df[candidate_numeric].mean()) / filtered_df[candidate_numeric].std()
print(f"Normalized {candidate_numeric} for filtered records:")
display(filtered_df[[candidate_numeric, norm_col]].head())

# Try grouping by a categorical field (not the index)
group_field = None
for col in filtered_df.columns:
    if col == candidate_numeric:
        continue
    if pd.api.types.is_object_dtype(filtered_df[col]):
        group_field = col
        break
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[candidate_numeric].mean().reset_index()
    print(f"Grouped data by {group_field} (mean of {candidate_numeric}):")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 4))
sns.histplot(data=filtered_df, x=candidate_numeric, bins=30, kde=True)
plt.title(f'Distribution of {candidate_numeric} (Filtered)')
plt.xlabel(candidate_numeric)
plt.ylabel('Count')
plt.show()

# Optional: Boxplot by group if available
if group_field is not None:
    plt.figure(figsize=(12, 5))
    sns.boxplot(data=filtered_df, x=group_field, y=candidate_numeric)
    plt.title(f'{candidate_numeric} by {group_field}')
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR^2 mass spectrometry analysis dataset using `mlcroissant`, explored its record set structure using their `@id`, extracted data into DataFrames, performed basic exploratory data analysis including normalization and grouping, and visualized core numeric field distributions.

You can further explore the various fields and record sets for deeper biological and statistical analysis relevant to extracellular vesicle proteomics.